# King - Man + Woman = Queen

We check if our system has similar properties

In [1]:
from similarity import load_eval_sentences_cached, load_og_sentences
from utils import DATA_DIR
import os
from tucker_tensor import TuckerDecomposition

dataset="fineweb-en"
random_state = 1
# we load the sample sentences only once
vector_path = os.path.join(DATA_DIR, "vectors", "fineweb_english_vectors.csv")
sentence_sample = load_eval_sentences_cached(vector_path=vector_path,
                                             dataset=dataset,
                                             seed=random_state,
                                             n_samples=10_000,
                                             )
full_dataset = load_og_sentences(vector_path=vector_path)
clean_sample = []
for vector in sentence_sample:
    cleaned = []
    for i, element in enumerate(vector):
        if element == "~":
            cleaned.append("None")
        else:
            cleaned.append(element)
    clean_sample.append(tuple(cleaned))
tucker = TuckerDecomposition.load_from_disk(
    dataset=dataset,
    method="siiSoftPlus",
    divergence="fr",
    dims=1000,
    rank=150,
    iterations=500
)


2026-02-12 16:41:38.625819: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
for el in ["king", "man", "woman", "queen"]:
    print(el, "in vocab:", el in tucker.vocab["vocab_s"])

king in vocab: False
man in vocab: True
woman in vocab: True
queen in vocab: False


In [18]:
from tucker_tensor import ExtendedTucker
# extended = ExtendedTucker.extend_tucker(
#     tucker,
#     dataset=full_dataset,
#     roles=["verb", "subject", "object"],
#     min_count=50,
#     fraction_threads=0.66,
#     normalize=True,
#
# )
# extended.save_extensions(DATA_DIR/"tensors"/dataset/"extensions/extension_test.pt")
new_extended = ExtendedTucker.load_extensions(tucker, DATA_DIR/"tensors"/dataset/"extensions/extension_norm_test.pt",
)

In [19]:

for el in ["king", "man", "woman", "queen"]:
    print(el, "in vocab:", el in new_extended.extensions["subject"])

king in vocab: True
man in vocab: False
woman in vocab: False
queen in vocab: True


In [20]:
integrated = new_extended.integrate_extension(None)

{'verb': 369, 'subject': 2475, 'object': 2625}


In [21]:
test_set = ["king", "man", "woman", "queen"]
for el in test_set:
    print(el, "in vocab:", el in integrated.vocab["vocab_s"])

king in vocab: True
man in vocab: True
woman in vocab: True
queen in vocab: True


In [49]:
from tucker_tensor import np_sim
import numpy as np
# we calculate the similarity between the elements
test_set = ["Paris", "France", "Washington", "US"]
for el in test_set:
    print(el, "in vocab:", el in tucker.vocab["vocab_s"])
for i, el in enumerate(test_set):
    lat = integrated.fetch_single_latent(el, "subject")
    for j, el2 in enumerate(test_set):
        lat2 = integrated.fetch_single_latent(el2, "subject")
        print(f"similarity {el}-{el2} = {np_sim(lat, lat2)}")


Paris in vocab: False
France in vocab: False
Washington in vocab: False
US in vocab: True
similarity Paris-Paris = 1.0
similarity Paris-France = -0.016994735226035118
similarity Paris-Washington = 0.17049524188041687
similarity Paris-US = 0.1337527483701706
similarity France-Paris = -0.016994735226035118
similarity France-France = 0.9999999403953552
similarity France-Washington = 0.2721351981163025
similarity France-US = 0.14683355391025543
similarity Washington-Paris = 0.17049524188041687
similarity Washington-France = 0.2721351981163025
similarity Washington-Washington = 1.0
similarity Washington-US = 0.37235286831855774
similarity US-Paris = 0.1337527483701706
similarity US-France = 0.14683355391025543
similarity US-Washington = 0.37235286831855774
similarity US-US = 0.9999999403953552


In [24]:
from tucker_tensor import np_sim
import numpy as np
# we calculate the similarity between the elements
test_set = ["king", "man", "woman", "queen"]

for i, el in enumerate(test_set):
    lat = integrated.fetch_single_latent(el, "subject")
    # we normalise this to 0-1
    for j, el2 in enumerate(test_set):
        lat2 = integrated.fetch_single_latent(el2, "subject")
        print(f"similarity {el}-{el2} = {np_sim(lat, lat2)}")


similarity king-king = 1.0
similarity king-man = 0.012082293629646301
similarity king-woman = 0.004348682705312967
similarity king-queen = 0.06405648589134216
similarity man-king = 0.012082293629646301
similarity man-man = 1.0000001192092896
similarity man-woman = 0.9036183953285217
similarity man-queen = 0.14416056871414185
similarity woman-king = 0.004348682705312967
similarity woman-man = 0.9036183953285217
similarity woman-woman = 1.0
similarity woman-queen = 0.06062542647123337
similarity queen-king = 0.06405648589134216
similarity queen-man = 0.14416056871414185
similarity queen-woman = 0.06062542647123337
similarity queen-queen = 1.0


In [31]:
test_set = ["king", "man", "woman", "queen"]

lats = {el: integrated.fetch_single_latent(el, "subject") for el in test_set}
additive_lat = lats["king"] - lats["man"] + lats["woman"]
multiplicative_lat = lats["king"] / lats["man"] * lats["woman"]
print(f"similarity king - queen = {np_sim(lats["king"], lats["queen"])}")
print(f"similarity additive element - queen = {np_sim(additive_lat, lats["queen"])}")
print(f"similarity multiplicative element - queen = {np_sim(multiplicative_lat, lats["queen"])}")


similarity king - queen = 0.06405648589134216
similarity additive element - queen = -0.05181814730167389
similarity multiplicative element - queen = -0.037530187517404556


In [30]:
# trying with the regular tucker
test_set = ["Paris", "France", "Washington", "US"]
for el in test_set:
    print(el, "in vocab:", el in integrated.vocab["vocab_s"])
lats = {el: integrated.fetch_single_latent(el, "subject") for el in test_set}
additive_lat = lats["Paris"] - lats["France"] + lats["Washington"]
multiplicative_lat = lats["Paris"] / lats["France"] * lats["Washington"]
print(f"similarity France - US = {np_sim(lats["France"], lats["US"])}")
print(f"similarity additive element - US = {np_sim(additive_lat, lats["US"])}")
print(f"similarity multiplicative element - US = {np_sim(multiplicative_lat, lats["US"])}")

Paris in vocab: True
France in vocab: True
Washington in vocab: True
US in vocab: True
similarity France - US = 0.14683355391025543
similarity additive element - US = 0.1814957559108734
similarity multiplicative element - US = 0.07810178399085999


In [32]:
# update to avoid division by 0
F = integrated.factors[1].cpu().numpy()  # (N,R)

# --- defensive norm computation ---
F_norm = np.linalg.norm(F, axis=1)
G_norm = np.linalg.norm(additive_lat)

eps = 1e-12  # safeguard lower bound
F_norm = np.maximum(F_norm, eps)
G_norm = max(G_norm, eps)

# --- safe cosine similarities ---
similarities = (F @ additive_lat) / (F_norm * G_norm)
top_k_indices = np.argsort(similarities)[-5:][::-1]
results = []
for idx in top_k_indices:
    role_str = next(k for k, v in integrated.vocab["s2i"].items() if v == idx)
    print(role_str)

king
spider
they
survivor
bear


In [33]:
# update to avoid division by 0
F = integrated.factors[1].cpu().numpy()  # (N,R)

# --- defensive norm computation ---
F_norm = np.linalg.norm(F, axis=1)
G_norm = np.linalg.norm(lats["king"])

eps = 1e-12  # safeguard lower bound
F_norm = np.maximum(F_norm, eps)
G_norm = max(G_norm, eps)

# --- safe cosine similarities ---
similarities = (F @ additive_lat) / (F_norm * G_norm)
top_k_indices = np.argsort(similarities)[-5:][::-1]
results = []
for idx in top_k_indices:
    role_str = next(k for k, v in integrated.vocab["s2i"].items() if v == idx)
    print(role_str)

king
spider
they
survivor
bear


In [46]:
test_set = ["king", "man", "woman", "queen"]

lats = {el: integrated.fetch_single_latent(el, "subject") for el in test_set}
additive_lat = lats["man"] + lats["king"] + lats["woman"]
multiplicative_lat = lats["man"] * lats["woman"] * lats["king"]
print(f"similarity king - queen = {np_sim(lats["king"], lats["queen"])}")
print(f"similarity additive element - queen = {np_sim(additive_lat, lats["queen"])}")
print(f"similarity multiplicative element - queen = {np_sim(multiplicative_lat, lats["queen"])}")


similarity king - queen = 0.06405648589134216
similarity additive element - queen = 0.12053338438272476
similarity multiplicative element - queen = 0.021176083013415337


In [47]:
test_set = ["king", "man", "woman", "queen"]

lats = {el: integrated.fetch_single_latent(el, "subject") for el in test_set}
additive_lat = lats["king"] - lats["man"] + lats["woman"]
multiplicative_lat = lats["king"] / lats["man"] * lats["woman"]
print(f"similarity king - queen = {np_sim(lats["king"], lats["queen"])}")
print(f"similarity additive element - queen = {np_sim(additive_lat, lats["queen"])}")
print(f"similarity multiplicative element - queen = {np_sim(multiplicative_lat, lats["queen"])}")


similarity king - queen = 0.06405648589134216
similarity additive element - queen = -0.05181814730167389
similarity multiplicative element - queen = -0.037530187517404556


In [20]:
tucker = TuckerDecomposition.load_from_disk(
    dataset=dataset,
    method="siiSoftPlus",
    name="largedim",
    divergence="fr",
    dims=10000,
    rank=150,
    iterations=500
)

In [21]:
from tucker_tensor import np_sim
import numpy as np
# we calculate the similarity between the elements
test_set = ["king", "man", "woman", "queen"]

for i, el in enumerate(test_set):
    lat = tucker.fetch_single_latent(el, "subject")
    # we normalise this to 0-1
    for j, el2 in enumerate(test_set):
        lat2 = tucker.fetch_single_latent(el2, "subject")
        print(f"similarity {el}-{el2} = {np_sim(lat, lat2)}")


similarity king-king = 0.9999999403953552
similarity king-man = 0.413642942905426
similarity king-woman = 0.3083689212799072
similarity king-queen = 0.5607380270957947
similarity man-king = 0.413642942905426
similarity man-man = 1.0
similarity man-woman = 0.43113166093826294
similarity man-queen = 0.19748036563396454
similarity woman-king = 0.3083689212799072
similarity woman-man = 0.43113166093826294
similarity woman-woman = 0.9999998807907104
similarity woman-queen = 0.19391484558582306
similarity queen-king = 0.5607380270957947
similarity queen-man = 0.19748036563396454
similarity queen-woman = 0.19391484558582306
similarity queen-queen = 1.0000001192092896


In [41]:
test_set = ["king", "man", "woman", "queen"]

lats = {el: tucker.fetch_single_latent(el, "subject") for el in test_set}
additive_lat = lats["king"] - lats["man"] + lats["woman"]
multiplicative_lat = lats["man"] * lats["woman"] * lats["king"]
print(f"similarity king - queen = {np_sim(lats["king"], lats["queen"])}")
print(f"similarity additive element - queen = {np_sim(additive_lat, lats["queen"])}")
print(f"similarity multiplicative element - queen = {np_sim(multiplicative_lat, lats["queen"])}")


similarity king - queen = 0.5607380270957947
similarity additive element - queen = 0.17458246648311615
similarity multiplicative element - queen = 0.13809818029403687


# Vecto package


In [24]:
# We will try to use our embedding matrices as vector lookup tables
tucker.factors[1].shape

torch.Size([10000, 150])

In [25]:
import numpy as np
from pathlib import Path
goal_path = Path("/home/local/stefa/data/tensors/fineweb-en/vecto/largedim_20K_fr_vsm")
np.save(goal_path / "subjects.npy", tucker.factors[1])

In [26]:
tucker.vocab.keys()

dict_keys(['vocab_v', 'vocab_s', 'vocab_o', 'v2i', 's2i', 'o2i'])

In [27]:
voc = tucker.vocab["vocab_s"]

In [28]:
with open(goal_path / "subjects.vocab", "w", encoding="utf-8") as f:
    for el in voc:
        # one token per line; strip newlines just in case
        f.write(str(el).replace("\n", " ").strip() + "\n")

In [29]:
from vecto.embeddings import load_from_dir

vsm = load_from_dir(goal_path)

In [30]:
vsm.normalize()

In [31]:
vsm.get_most_similar_words("cybercriminal", cnt=5)

[['cybercriminal', np.float32(1.0)],
 ['scammer', np.float32(0.916242)],
 ['radiologist', np.float32(0.9081334)],
 ['soundbar', np.float32(0.8981118)],
 ['cleaner', np.float32(0.89306784)]]

In [39]:
vsm.get_most_similar_words("scammer", cnt=5)

[['scammer', np.float32(1.0)],
 ['cybercriminal', np.float32(0.916242)],
 ['criminal', np.float32(0.8739346)],
 ['Qualcomm', np.float32(0.8701472)],
 ['radiologist', np.float32(0.8621169)]]